# Clase 012 — Logging

**Parte 0** · Logging HOWTO.

> 🎯 Dejar `print` para debug y usar `logging` con niveles, handlers, formatters. La diferencia entre código observable y código que adivinas.

> ⏱️ ~60 min

## ⚙️ Setup

In [ ]:
import logging
import tempfile
from pathlib import Path
from logging.config import dictConfig

## 1️⃣ Por qué `logging` y no `print`

| `print` | `logging` |
|---|---|
| stdout fijo | múltiples destinos |
| sin nivel | DEBUG/INFO/WARNING/ERROR/CRITICAL |
| sin contexto | módulo, función, timestamp automáticos |
| no se silencia sin tocar código | filtras por nivel |
| no estructurado | parseable, integrable con observabilidad |

## 2️⃣ Niveles

| Nivel | Uso |
|---|---|
| `DEBUG` | detalles para diagnóstico (variables, flujo) |
| `INFO` | progreso normal ("cargados 1000 registros") |
| `WARNING` | algo raro pero no fatal ("valor por defecto usado") |
| `ERROR` | la operación falló ("no se pudo cargar el CSV") |
| `CRITICAL` | sistema no puede continuar |

Filtro: si configuras nivel `INFO`, solo se muestran INFO/WARNING/ERROR/CRITICAL.

## 3️⃣ Setup mínimo con `basicConfig`

```python
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
)
log = logging.getLogger(__name__)
log.info('arrancando...')
log.warning('cuidado')
```

⚠️ **Gotcha**: `basicConfig` solo aplica si **no había handlers** en root. En Jupyter (kernel reusado) puede no tener efecto — usa `force=True`.

In [ ]:
# Reset y configuración explícita para notebook
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)

log = logging.getLogger(__name__)
log.debug('detalle interno')
log.info('todo OK')
log.warning('algo raro')
log.error('fallo manejable')

## 4️⃣ Logger por módulo — la práctica correcta

```python
# loader.py
import logging
log = logging.getLogger(__name__)   # 'loader' o 'mi_pkg.loader'

def cargar(path):
    log.info(f'cargando {path}')
    ...
```

Ventaja: el root logger configurado **una vez** propaga a todos los módulos. Puedes silenciar uno solo con `logging.getLogger('loader').setLevel(WARNING)`.

In [ ]:
# Simula 2 módulos
log_app = logging.getLogger('mi_app')
log_db  = logging.getLogger('mi_app.db')

log_app.info('arrancando app')
log_db.info('conectando a db')
log_db.warning('latencia alta')

# Silencia un módulo específico
log_db.setLevel(logging.ERROR)
log_db.info('esto NO se ve')
log_db.error('esto sí se ve')

## 5️⃣ Handler doble — consola + archivo

Para producción típicamente queremos:
- Consola: INFO+ (lo que el operador ve)
- Archivo: DEBUG+ (todo para post-mortem)

In [ ]:
log_file = Path(tempfile.gettempdir()) / 'demo_app.log'

config = {
    'version': 1,
    'disable_existing_loggers': False,
    'formatters': {
        'verbose': {'format': '%(asctime)s [%(levelname)s] %(name)s: %(message)s'},
        'corto':   {'format': '[%(levelname)s] %(message)s'},
    },
    'handlers': {
        'consola': {
            'class': 'logging.StreamHandler',
            'level': 'INFO',
            'formatter': 'corto',
        },
        'archivo': {
            'class': 'logging.FileHandler',
            'filename': str(log_file),
            'level': 'DEBUG',
            'formatter': 'verbose',
            'mode': 'w',
        },
    },
    'root': {
        'level': 'DEBUG',
        'handlers': ['consola', 'archivo'],
    },
}
dictConfig(config)

log = logging.getLogger('demo')
log.debug('DEBUG: solo va al archivo')
log.info('INFO: va a ambos')
log.warning('WARN: va a ambos')
log.error('ERROR: va a ambos')

print(f'\n--- contenido de {log_file} ---')
print(log_file.read_text())

## 6️⃣ Buenas prácticas

- **No hagas `f'{var}'` en el mensaje** si vas a filtrar por nivel: pasa args separados, `log.debug('valor: %s', var)`. Así no se formatea si el nivel no aplica.
- **No loguees datos sensibles**: PII, tokens, contraseñas. Filtra antes.
- **`exc_info=True`** en `log.error` para capturar el traceback completo.
- **Logger por módulo**, configuración por aplicación. No mezcles.

In [ ]:
# exc_info=True para capturar traceback
try:
    1 / 0
except ZeroDivisionError:
    log.error('división falló', exc_info=True)

## ✅ Checklist

- [ ] Sé los 5 niveles y cuándo usar cada uno
- [ ] Uso `getLogger(__name__)` en cada módulo
- [ ] Configuro logging UNA vez en el entrypoint
- [ ] Tengo handler consola (INFO+) y archivo (DEBUG+)
- [ ] Uso `exc_info=True` para errores con stacktrace

## 📝 Homework

Ver `README.md`. Notebook + 2 módulos + `logging_config.py` con `dictConfig`; entrega `app.log`.

## 🔗 Referencias

- [Logging HOWTO](https://docs.python.org/3/howto/logging.html)
- [Logging Cookbook](https://docs.python.org/3/howto/logging-cookbook.html)

➡️ **Siguiente:** [013 — Type hints y mypy](../013-type-hints-y-mypy/README.md)